# 🌊 Notebook 1 — FathomNet: Download & Build Database

**Goal**: Download FathomNet from Zenodo Mini, crop images based on YOLO bounding boxes, enrich with API metadata, and publish as Kaggle Dataset.

## 0. Install & import

In [ ]:
!pip install requests tqdm pillow pandas pyyaml -q

In [ ]:
import os, json, zipfile, sqlite3, shutil, hashlib, time, re
from pathlib import Path
from typing import Optional, List, Dict

import requests
from tqdm.auto import tqdm
import pandas as pd
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

WORK         = Path('/kaggle/working')
ZIP_PATH     = WORK / 'fathomnet_raw.zip'
EXTRACT_DIR  = WORK / 'fathomnet_raw'

BASE_OUT     = WORK / 'fathomnet_db'
IMG_DIR      = BASE_OUT / 'images'
META_DIR     = BASE_OUT / 'metadata'
DB_PATH      = BASE_OUT / 'fathomnet.db'
CSV_PATH     = META_DIR / 'metadata.csv'

for d in [IMG_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Output  : {BASE_OUT}')

## 1. Download & extract Zenodo ZIP

In [ ]:
ZENODO_ZIP_URL = ('https://zenodo.org/records/8032379/files/'
                  'Fathomnet-by-Phylum-2.zip?download=1')

def download_file(url: str, dest: Path, chunk_size: int = 1 << 20) -> None:
    if dest.exists():
        return
    print(f'Downloading {dest.name} ...')
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(dest, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as bar:
            for chunk in r.iter_content(chunk_size):
                f.write(chunk)
                bar.update(len(chunk))

download_file(ZENODO_ZIP_URL, ZIP_PATH)

In [ ]:
if not EXTRACT_DIR.exists():
    print('Extracting ...')
    with zipfile.ZipFile(ZIP_PATH) as zf:
        members = zf.namelist()
        for m in tqdm(members, desc='Extracting'):
            zf.extract(m, EXTRACT_DIR)


## 2. Parse YOLO dataset & Contextual Crop

In [ ]:
possible_roots = list(EXTRACT_DIR.rglob('data.yaml')) + list(EXTRACT_DIR.rglob('dataset.yaml'))
DATASET_ROOT = possible_roots[0].parent if possible_roots else EXTRACT_DIR

class_names: Dict[int, str] = {}
yaml_files = list(DATASET_ROOT.rglob('*.yaml'))
if yaml_files:
    try:
        import yaml
        with open(yaml_files[0]) as f:
            cfg = yaml.safe_load(f)
        class_names = {i: n for i, n in enumerate(cfg.get('names', []))}
    except Exception:
        pass

if not class_names:
    PHYLUM_FALLBACK = [
        'Annelida', 'Arthropoda', 'Bryozoa', 'Chordata', 'Cnidaria',
        'Echinodermata', 'Mollusca', 'Platyhelminthes', 'Porifera', 'other'
    ]
    class_names = {i: n for i, n in enumerate(PHYLUM_FALLBACK)}

print(f'Total classes found: {len(class_names)}')
print(list(class_names.values()))

In [ ]:
def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def get_split(p: Path) -> str:
    parts = [x.lower() for x in p.parts]
    for s in ['train', 'valid', 'val', 'test']: 
        if s in parts: return s if s != 'val' else 'valid'
    return 'train'

def extract_fathomnet_uuid(filename: str) -> Optional[str]:
    # Fathomnet UUID pattern: 8-4-4-4-12 hex chars
    m = re.search(r'([a-f0-9]{8}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{12})', filename.lower())
    return m.group(1) if m else None

def parse_labels(img_path: Path, cls_map: Dict[int, str]) -> List[Dict]:
    label_path = Path(str(img_path).replace('/images/', '/labels/').replace('\\images\\', '\\labels\\')).with_suffix('.txt')
    annots = []
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.split()
            if len(parts) >= 5:
                cid = int(parts[0])
                annots.append({
                    'class_id':   cid,
                    'class_name': cls_map.get(cid, f'class_{cid}'),
                    'cx': float(parts[1]), 'cy': float(parts[2]),
                    'w':  float(parts[3]), 'h':  float(parts[4]),
                })
    return annots

all_images = (sorted(DATASET_ROOT.rglob('*.jpg')) +
              sorted(DATASET_ROOT.rglob('*.jpeg')) +
              sorted(DATASET_ROOT.rglob('*.png')))
print(f'Found {len(all_images)} raw images')

seen_hashes = set()
records = []

MARGIN_RATIO = 0.3 # 30% padding
MIN_SIZE = 50

for img_path in tqdm(all_images, desc='Cropping images'):
    content_hash = sha256_of(img_path)[:16]
    if content_hash in seen_hashes:
        continue
    seen_hashes.add(content_hash)

    fathom_uuid = extract_fathomnet_uuid(img_path.name)
    annots = parse_labels(img_path, class_names)

    try:
        with Image.open(img_path) as im:
            width, height = im.size
            img_rgb = im.convert('RGB')
    except Exception:
        continue

    if not annots:
        out_name = f'{content_hash}_full.jpg'
        out_path = IMG_DIR / out_name
        if not out_path.exists():
            img_rgb.save(out_path, quality=95)
        records.append({
            'image_id':         content_hash,
            'filename':         out_name,
            'fathom_uuid':      fathom_uuid,
            'split':            get_split(img_path),
            'width':            width,
            'height':           height,
            'n_annotations':    0,
            'species':          'Unknown',
            'phyla':            'Unknown',
            'annotations_json': '[]',
        })
        continue

    for k, ann in enumerate(annots):
        cx, cy, w, h = ann['cx'], ann['cy'], ann['w'], ann['h']
        box_w, box_h = w * width, h * height
        x_center, y_center = cx * width, cy * height
        
        marg_w, marg_h = box_w * MARGIN_RATIO, box_h * MARGIN_RATIO
        x_min = max(0, int(x_center - box_w/2 - marg_w))
        y_min = max(0, int(y_center - box_h/2 - marg_h))
        x_max = min(width, int(x_center + box_w/2 + marg_w))
        y_max = min(height, int(y_center + box_h/2 + marg_h))
        
        crop_w = x_max - x_min
        crop_h = y_max - y_min
        
        if crop_w < MIN_SIZE or crop_h < MIN_SIZE:
            continue
            
        crop_id = f'{content_hash}_{k}'
        out_name = f'{crop_id}.jpg'
        out_path = IMG_DIR / out_name
        
        if not out_path.exists():
            img_rgb.crop((x_min, y_min, x_max, y_max)).save(out_path, quality=95)
            
        species_name = ann.get('class_name', '')
        phylum_name = class_names.get(ann.get('class_id', -1), '?')
        
        records.append({
            'image_id':         crop_id,
            'filename':         out_name,
            'fathom_uuid':      fathom_uuid,
            'split':            get_split(img_path),
            'width':            crop_w,
            'height':           crop_h,
            'n_annotations':    1,
            'species':          species_name,
            'phyla':            phylum_name,
            'annotations_json': json.dumps([ann]),
        })

df = pd.DataFrame(records)
print(f'Total cropped images: {len(df)}')


## 3. Enrich existing files using FathomNet API (NO Extra Images)

In [ ]:
FATHOMNET_API = 'https://fathomnet.org/fathomnet/v1'

unique_uuids = df['fathom_uuid'].dropna().unique()
print(f'Fetching metadata for {len(unique_uuids)} unique UUIDs directly from FathomNet...')

def safe_depth(raw) -> Optional[float]:
    if raw is None: return None
    if isinstance(raw, list):
        vals = [v for v in raw if v is not None]
        return float(sum(vals) / len(vals)) if vals else None
    try:    return float(raw)
    except: return None

uuid_meta = {}
for uid in tqdm(unique_uuids, desc='API Fetch'):
    try:
        r = requests.get(f'{FATHOMNET_API}/images/{uid}', timeout=10)
        if r.status_code == 200:
            data = r.json()
            uuid_meta[uid] = {
                'depth_m': safe_depth(data.get('depthMeters')),
                'temperature_c': data.get('temperatureCelsius'),
                'latitude': data.get('latitude'),
                'longitude': data.get('longitude'),
                'recorded_at': str(data.get('timestamp', '')),
                'url': data.get('url')
            }
    except Exception:
        pass

df['depth_m'] = df['fathom_uuid'].map(lambda x: uuid_meta.get(x, {}).get('depth_m'))
df['temperature_c'] = df['fathom_uuid'].map(lambda x: uuid_meta.get(x, {}).get('temperature_c'))
df['latitude'] = df['fathom_uuid'].map(lambda x: uuid_meta.get(x, {}).get('latitude'))
df['longitude'] = df['fathom_uuid'].map(lambda x: uuid_meta.get(x, {}).get('longitude'))
df['recorded_at'] = df['fathom_uuid'].map(lambda x: uuid_meta.get(x, {}).get('recorded_at'))
df['url'] = df['fathom_uuid'].map(lambda x: uuid_meta.get(x, {}).get('url'))

df_master = df.copy()
df_master['species_clean'] = df_master['species'].fillna('').str.strip().str.title()

print(f'Metadata enrichment completed!')
print(f"Rows with Depth info: {df_master['depth_m'].notna().sum()}")
print(f"Rows with Temp info: {df_master['temperature_c'].notna().sum()}")

## 4. Build SQLite database

In [ ]:
cols = [
    'image_id', 'filename', 'split', 'width', 'height', 'n_annotations',
    'species', 'phyla', 'species_clean', 'depth_m', 'temperature_c',
    'latitude', 'longitude', 'recorded_at', 'url', 'annotations_json'
]

conn   = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.executescript("""
CREATE TABLE IF NOT EXISTS image_text_pairs (
    pair_id    INTEGER PRIMARY KEY AUTOINCREMENT,
    image_id   TEXT NOT NULL,
    caption    TEXT,
    prompt_used TEXT,
    model_name  TEXT,
    created_at  TEXT DEFAULT (datetime('now'))
);
""")
conn.commit()

df_master[cols].to_sql('images', conn, if_exists='replace', index=False)

cursor.executescript("""
CREATE INDEX IF NOT EXISTS idx_images_species ON images(species_clean);
CREATE INDEX IF NOT EXISTS idx_images_split   ON images(split);
CREATE INDEX IF NOT EXISTS idx_pairs_img      ON image_text_pairs(image_id);
""")
conn.commit()

n = cursor.execute('SELECT COUNT(*) FROM images').fetchone()[0]
conn.close()

print(f'DB written -> {DB_PATH}  ({n} images, {DB_PATH.stat().st_size/1e6:.2f} MB)')

## 5. Save CSV + summary

In [ ]:
df_master.to_csv(CSV_PATH, index=False)
print(f'CSV -> {CSV_PATH}')

with open(META_DIR / 'class_names.json', 'w') as f:
    json.dump({str(k): v for k, v in class_names.items()}, f, indent=2)

has_depth = df_master['depth_m'].notna()
summary = {
    'total_images':      int(len(df_master)),
    'splits':            df_master['split'].value_counts().to_dict(),
    'unique_species':    int(df_master['species_clean'].nunique()),
    'images_with_depth': int(has_depth.sum())
}
(META_DIR / 'dataset_summary.json').write_text(json.dumps(summary, indent=2))

print('\n=== Dataset Summary ===')
for k, v in summary.items():
    print(f'  {k}: {v}')

## 6. Cleanup

In [ ]:
import shutil as _shutil
for tmp in [EXTRACT_DIR, ZIP_PATH]:
    if tmp.exists():
        if tmp.is_dir():
            _shutil.rmtree(tmp)
        else:
            tmp.unlink()
print('Cleanup complete. Ready to publish!')